# 19B — V5 Final DEV Event QA + Freeze

This notebook applies the immutable manual source-verification override and freezes the repaired DEV event corpus.

It **does not** pair events, inspect 1–5 year gaps, rebalance chronology, generate astrology, score Control, or research CONFIRM.

Success status:

`V5_DEV_EVENT_CORPUS_FROZEN_READY_FOR_LOCAL_PAIRING_NUISANCE_GATE`


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json
import pandas as pd
import numpy as np

NOTEBOOK_VERSION="SAJU_ML_V5_FINAL_EVENT_QA_FREEZE_20260817"

def find_repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repo.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

ROOT=find_repo_root()
SRC=ROOT/"research/ml/artifacts/v5_source_verification"
REPAIR=ROOT/"research/ml/artifacts/v5_identity_repair"
CORPUS=ROOT/"research/ml_corpus/v5_ground_truth"
OUT=ROOT/"research/ml/artifacts/v5_event_freeze"
OUT.mkdir(parents=True,exist_ok=True)

ASSEMBLED=SRC/"V5_DEV_EVENTS_REPAIRED_ASSEMBLED_PRE_SOURCE_QA.csv"
AUTO=SRC/"V5_SOURCE_VERIFICATION_RESULTS_AUTO.csv"
MANUAL=SRC/"V5_SOURCE_VERIFICATION_ELIGIBLE_MANUAL_REVIEW.csv"
GATE_DECISION=SRC/"V5_SOURCE_VERIFICATION_GATE_DECISION.json"
GATE_SUMMARY=SRC/"V5_SOURCE_VERIFICATION_GATE_SUMMARY.json"
OVERRIDE=SRC/"V5_SOURCE_VERIFICATION_MANUAL_OVERRIDE.csv"
OVERRIDE_SUMMARY=SRC/"V5_SOURCE_VERIFICATION_MANUAL_BACKFILL_SUMMARY.json"
ROSTER=REPAIR/"V5_DEV_SUBJECT_ROSTER_160_REPAIRED.csv"
PROTO=CORPUS/"V5_FINAL_EVENT_FREEZE_PROTOCOL.json"

for p in [ASSEMBLED,AUTO,MANUAL,GATE_DECISION,GATE_SUMMARY,OVERRIDE,OVERRIDE_SUMMARY,ROSTER,PROTO]:
    if not p.exists(): raise FileNotFoundError(p)

gate=json.load(open(GATE_DECISION,encoding="utf-8"))
gsummary=json.load(open(GATE_SUMMARY,encoding="utf-8"))
osummary=json.load(open(OVERRIDE_SUMMARY,encoding="utf-8"))
proto=json.load(open(PROTO,encoding="utf-8"))

assert gate["status"]=="V5_SOURCE_VERIFICATION_MANUAL_BACKFILL_REQUIRED"
assert proto["status"]=="PREDECLARED_AFTER_SOURCE_BACKFILL_BEFORE_EVENT_FREEZE"
assert sha256_file(ASSEMBLED)==gate["assembled_events_sha256"]
assert sha256_file(AUTO)==gate["auto_results_sha256"]
assert sha256_file(MANUAL)==gate["eligible_manual_review_sha256"]
assert sha256_file(GATE_SUMMARY)==gate["summary_sha256"]
assert sha256_file(MANUAL)==osummary["manual_review_input_sha256"]
assert sha256_file(OVERRIDE)==osummary["manual_override_sha256"]
assert osummary["status"]=="V5_MANUAL_SOURCE_BACKFILL_COMPLETE_ALL_ELIGIBLE_ROWS_VERIFIED"

events=pd.read_csv(ASSEMBLED)
auto=pd.read_csv(AUTO)
manual=pd.read_csv(MANUAL)
override=pd.read_csv(OVERRIDE)
roster=pd.read_csv(ROSTER)

print("19B PREFLIGHT PASS")


19B PREFLIGHT PASS


## 1. Apply source-verification override by immutable `event_row_id`

In [2]:

assert events.event_row_id.is_unique
assert auto.event_row_id.is_unique
assert manual.event_row_id.is_unique
assert override.event_row_id.is_unique

assert set(manual.event_row_id)==set(override.event_row_id)
assert len(manual)==118
assert len(override)==118

verified_manual_status={"VERIFIED_ORIGINAL","VERIFIED_ALTERNATE"}
assert set(override.manual_verification_status).issubset(verified_manual_status)
assert override.event_fact_verified.astype(bool).all()
assert override.polarity_mechanism_supported.astype(bool).all()
assert (~override.event_correction_required.astype(bool)).all()

auto_status=auto.set_index("event_row_id")["verification_status"].to_dict()
manual_status=override.set_index("event_row_id")["manual_verification_status"].to_dict()

def final_verification_status(rid):
    a=auto_status.get(rid)
    if a=="AUTO_PASS":
        return "VERIFIED_AUTO"
    if rid in manual_status:
        return manual_status[rid]
    return a or "UNVERIFIED"

events["final_source_verification"]=events.event_row_id.map(final_verification_status)

eligible=events[~events.exclude.astype(bool)].copy()
excluded=events[events.exclude.astype(bool)].copy()

verified_values={"VERIFIED_AUTO","VERIFIED_ORIGINAL","VERIFIED_ALTERNATE"}
unverified_eligible=eligible[~eligible.final_source_verification.isin(verified_values)]
assert len(unverified_eligible)==0, unverified_eligible[
    ["event_row_id","subject_id","event_year","source_url","final_source_verification"]
].to_dict(orient="records")[:20]

print("Eligible source verification coverage: 100%")
print(eligible.final_source_verification.value_counts())


Eligible source verification coverage: 100%
VERIFIED_AUTO         287
VERIFIED_ORIGINAL     110
VERIFIED_ALTERNATE      8
Name: final_source_verification, dtype: int64


## 2. Structural integrity QA against repaired roster

In [3]:

assert len(events)==gsummary["event_rows_n"]==481
assert len(eligible)==gsummary["eligible_rows_n"]==405
assert len(excluded)==gsummary["excluded_rows_n"]==76
assert events.subject_id.nunique()==160
assert roster.subject_id.nunique()==160
assert set(events.subject_id)==set(roster.subject_id)

# One immutable axis per subject.
axis_n=events.groupby("subject_id").preassigned_axis.nunique()
assert axis_n.max()==1

roster_axis=roster.set_index("subject_id")["preassigned_axis"]
event_axis=events.groupby("subject_id").preassigned_axis.first()
assert (event_axis.sort_index()==roster_axis.loc[event_axis.index].sort_index()).all()

assert set(eligible.polarity.unique()).issubset({"positive","negative"})
assert eligible.event_year.notna().all()
assert eligible.event_description.fillna("").str.strip().ne("").all()
assert eligible.source_url.fillna("").str.startswith(("http://","https://")).all()

# This is not pairability analysis: report same-year opposite-polarity collisions only
# so they remain explicitly retained into the freeze.
yrpol=(
    eligible.groupby(["subject_id","event_year"]).polarity
    .agg(lambda x:set(x))
)
same_year_opposite=yrpol[yrpol.map(lambda s: {"positive","negative"}.issubset(s))]

print("STRUCTURAL QA PASS")
print("same-year opposite-polarity subject-years retained:",len(same_year_opposite))


STRUCTURAL QA PASS
same-year opposite-polarity subject-years retained: 21


## 3. Duplicate QA without deleting or rebalancing anything

In [4]:

# A true duplicate candidate requires same subject/year/polarity/type/description.
dup_key=["subject_id","event_year","polarity","event_type","event_description"]
dup_mask=eligible.duplicated(dup_key,keep=False)
dup=eligible[dup_mask].sort_values(dup_key).copy()

dup_path=OUT/"V5_DEV_EVENT_DUPLICATE_REVIEW_CANDIDATES.csv"
dup.to_csv(dup_path,index=False)

if len(dup):
    raise RuntimeError(
        f"Final freeze blocked: {len(dup)} eligible rows form exact semantic duplicate candidates. "
        "Review V5_DEV_EVENT_DUPLICATE_REVIEW_CANDIDATES.csv without using chronology/pairability."
    )

print("DUPLICATE QA PASS: 0 exact semantic duplicate candidates")


DUPLICATE QA PASS: 0 exact semantic duplicate candidates


## 4. Freeze the full audit corpus and eligible modeling corpus

In [5]:

sort_cols=[
    "subject_id","event_year","polarity","event_type","event_row_id"
]
full_frozen=events.sort_values(sort_cols,kind="stable").reset_index(drop=True)
eligible_frozen=eligible.sort_values(sort_cols,kind="stable").reset_index(drop=True)

full_path=OUT/"V5_DEV_EVENT_CORPUS_FULL_FROZEN.csv"
eligible_path=OUT/"V5_DEV_EVENT_CORPUS_ELIGIBLE_FROZEN.csv"
verification_path=OUT/"V5_DEV_EVENT_SOURCE_VERIFICATION_FINAL.csv"

full_frozen.to_csv(full_path,index=False)
eligible_frozen.to_csv(eligible_path,index=False)

verification_cols=[
    "event_row_id","batch_id","subject_id","name","event_year","polarity",
    "event_type","source_url","final_source_verification"
]
verification=full_frozen[verification_cols].copy()
ovcols=override[[
    "event_row_id","verified_source_url","verification_secondary_url",
    "source_grade","verification_note","event_correction_required"
]]
verification=verification.merge(ovcols,on="event_row_id",how="left",validate="one_to_one")
verification.to_csv(verification_path,index=False)

coverage=(
    eligible_frozen.groupby(["preassigned_axis","polarity"])
    .size().rename("eligible_event_rows").reset_index()
)
subject_coverage=(
    eligible_frozen.groupby(["subject_id","preassigned_axis"])
    .agg(
        positive_events=("polarity",lambda x:int((x=="positive").sum())),
        negative_events=("polarity",lambda x:int((x=="negative").sum())),
        distinct_event_years=("event_year","nunique")
    ).reset_index()
)

coverage_path=OUT/"V5_DEV_EVENT_FREEZE_AXIS_POLARITY_COVERAGE.csv"
subject_coverage_path=OUT/"V5_DEV_EVENT_FREEZE_SUBJECT_COVERAGE.csv"
coverage.to_csv(coverage_path,index=False)
subject_coverage.to_csv(subject_coverage_path,index=False)

manifest={
    "version":"V5_DEV_EVENT_FREEZE_MANIFEST_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_DEV_EVENT_CORPUS_FROZEN_READY_FOR_LOCAL_PAIRING_NUISANCE_GATE",
    "DEV_subjects_n":160,
    "full_event_rows_n":int(len(full_frozen)),
    "eligible_event_rows_n":int(len(eligible_frozen)),
    "excluded_audit_rows_n":int(len(full_frozen)-len(eligible_frozen)),
    "eligible_source_verified_n":int(
        eligible_frozen.final_source_verification.isin(
            {"VERIFIED_AUTO","VERIFIED_ORIGINAL","VERIFIED_ALTERNATE"}
        ).sum()
    ),
    "same_year_opposite_polarity_subject_years_retained_n":int(len(same_year_opposite)),
    "full_corpus_sha256":sha256_file(full_path),
    "eligible_corpus_sha256":sha256_file(eligible_path),
    "source_verification_final_sha256":sha256_file(verification_path),
    "axis_polarity_coverage_sha256":sha256_file(coverage_path),
    "subject_coverage_sha256":sha256_file(subject_coverage_path),
    "repaired_DEV_roster_sha256":sha256_file(ROSTER),
    "manual_override_sha256":sha256_file(OVERRIDE),
    "freeze_protocol_sha256":sha256_file(PROTO),
    "rules":{
        "membership_changed":False,
        "event_rows_deleted_for_balance":False,
        "event_labels_changed":False,
        "event_years_changed":False,
        "axis_changed":False,
        "pairing_performed":False,
        "pair_gap_inspected":False,
        "chronology_balancing_performed":False,
        "astrology_generated":False,
        "control_scored":False,
        "confirm_researched":False
    },
    "pairing_may_begin":True,
    "astrology_generation_allowed":False,
    "confirm_event_research_allowed":False,
    "next_rule":(
        "Build all eligible same-subject same-axis opposite-polarity distinct-year pairs "
        "with abs(year gap) 1-5. Then run chronology and age×axis nuisance gates. "
        "No astrology until that gate passes."
    )
}

manifest_path=OUT/"V5_DEV_EVENT_FREEZE_MANIFEST.json"
json.dump(manifest,open(manifest_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

decision={
    "version":"V5_DEV_EVENT_FREEZE_DECISION_V1",
    "status":manifest["status"],
    "full_corpus_sha256":manifest["full_corpus_sha256"],
    "eligible_corpus_sha256":manifest["eligible_corpus_sha256"],
    "manifest_sha256":sha256_file(manifest_path),
    "pairing_may_begin":True,
    "astrology_generation_allowed":False,
    "confirm_event_research_allowed":False
}
decision_path=OUT/"V5_DEV_EVENT_FREEZE_DECISION.json"
json.dump(decision,open(decision_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps(manifest,ensure_ascii=False,indent=2))


{
  "version": "V5_DEV_EVENT_FREEZE_MANIFEST_V1",
  "notebook_version": "SAJU_ML_V5_FINAL_EVENT_QA_FREEZE_20260817",
  "created_at": "2026-08-17T04:52:17",
  "status": "V5_DEV_EVENT_CORPUS_FROZEN_READY_FOR_LOCAL_PAIRING_NUISANCE_GATE",
  "DEV_subjects_n": 160,
  "full_event_rows_n": 481,
  "eligible_event_rows_n": 405,
  "excluded_audit_rows_n": 76,
  "eligible_source_verified_n": 405,
  "same_year_opposite_polarity_subject_years_retained_n": 21,
  "full_corpus_sha256": "1750c3481ba5f5a957c713b89bff4b58648e3c7f1706d1867795575640689fe4",
  "eligible_corpus_sha256": "a125435a67afc56539b6bdebdf3c56ab02f7a551e472da3026454c51e2e06e87",
  "source_verification_final_sha256": "f387e849cc2bd7bce99f2f82646a85387f4d4dc8496566118afd457a83961d03",
  "axis_polarity_coverage_sha256": "e757b64e07712f6465610c34a96b60352c29dd2c48ae01b9aa928ec19cf57446",
  "subject_coverage_sha256": "9a808cb97115e2143ceacb520a600c82b44b75b09347f9d335aba6897bba4ace",
  "repaired_DEV_roster_sha256": "f40e8fc2021be911ef2a15

## Return to ChatGPT

If the notebook finishes with:

`V5_DEV_EVENT_CORPUS_FROZEN_READY_FOR_LOCAL_PAIRING_NUISANCE_GATE`

send exactly:

```text
V5_DEV_EVENT_FREEZE_DECISION.json
V5_DEV_EVENT_FREEZE_MANIFEST.json
V5_DEV_EVENT_FREEZE_AXIS_POLARITY_COVERAGE.csv
V5_DEV_EVENT_FREEZE_SUBJECT_COVERAGE.csv
```

Do **not** send or inspect pair results yet because this notebook does not create them.

The next notebook is Notebook 20: deterministic local pair generation + chronology/age×axis nuisance gate. Astrology remains prohibited until Notebook 20 passes.
